In [1]:
import autogen

##### Writer Agent

In [2]:
config_list_gemini = autogen.config_list_from_json("model_config.json")
writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config={"config_list" : config_list_gemini},
)

##### Setting up the task 

In [3]:
task = '''
        Write a concise but engaging blogpost about
       Devops. Make sure the blogpost is
       within 100 words.
       '''

In [4]:
reply = writer.generate_reply(messages=[{"content": task, "role": "user"}])

#### Critic agent

In [6]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config={"config_list" : config_list_gemini},
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [7]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       Devops. Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

## DevOps: Bridging the Gap

DevOps isn't just a buzzword; it's a cultural shift.  It's about breaking down silos between development and operations teams, fostering collaboration to deliver software faster and more reliably.  Through automation, continuous integration/continuous delivery (CI/CD), and shared responsibility, DevOps streamlines the entire software lifecycle, leading to quicker releases, improved quality, and happier users.  It's about agility and efficiency in the modern software world.


--------------------------------------------------------------------------------
Critic (to Writer):

This is a good start to a short blog post about DevOps, hitting the key points concisely.  However, it could be improved by adding a bit m

##### SEO Agent

In [8]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO Reviewer",
    llm_config={"config_list" : config_list_gemini},
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)


##### Legal Agent 

In [9]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal Reviewer",
    llm_config={"config_list" : config_list_gemini},
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

##### Ethics_reviewer Agent

In [10]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics Reviewer",
    llm_config={"config_list" : config_list_gemini},
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

##### Meta Reviewer

In [11]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta Reviewer",
    llm_config={"config_list" : config_list_gemini},
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

In [12]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]


In [ ]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

In [14]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       Devops. Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

## DevOps: Bridging the Gap

DevOps isn't just a buzzword; it's a cultural shift.  It's about breaking down silos between development and operations teams, fostering collaboration to deliver software faster and more reliably.  Through automation, continuous integration/continuous delivery (CI/CD), and shared responsibility, DevOps streamlines the entire software lifecycle, leading to quicker releases, improved quality, and happier users.  It's about agility and efficiency in the modern software world.


--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

****************************************************

In [15]:
print(res.summary)

## DevOps: Accelerating Software Delivery with Collaboration and Automation

DevOps is a set of practices, tools, and a cultural philosophy that automates and integrates the processes between software development and IT operations teams.  This collaboration aims to shorten the systems development life cycle and provide continuous delivery with high software quality.  Key DevOps principles include continuous integration (CI), continuous delivery (CD), and continuous deployment, often facilitated by CI/CD pipelines.  Popular DevOps tools include Jenkins, GitLab CI, Docker, Kubernetes, and Terraform, each playing a crucial role in automating different stages of the software delivery process.  Effective DevOps implementation also requires robust monitoring and logging solutions for efficient troubleshooting and performance optimization.  Exploring DevOps best practices, such as infrastructure as code (IaC) and DevSecOps (integrating security throughout the development lifecycle), is essent